In [45]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
import kagglehub
from kagglehub import KaggleDatasetAdapter, dataset_load

file_path = "student_performance_dataset.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "harshadapatil31/student-performance-and-study-habits-dataset",
  file_path
)
df.head()


Using Colab cache for faster access to the 'student-performance-and-study-habits-dataset' dataset.


,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,4,Male,2.6,77.5,8.0,NaN,Yes,Yes,No,85.1,83.8,B
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D


In [46]:
df.tail()

,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
995,996,Male,2.8,75.4,8.2,Masters,Yes,Yes,No,60.5,70.7,C
996,997,Male,6.7,88.4,7.1,NaN,No,Yes,No,82.2,99.5,A
997,998,Female,2.6,84.5,8.0,High School,Yes,Yes,Yes,65.2,79.2,C
998,999,Female,4.6,85.3,8.1,High School,No,No,Yes,52.2,82.2,B
999,1000,Male,3.9,77.4,6.1,Masters,Yes,No,No,45.4,70.4,C


In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   student_id                  1000 non-null   int64  
 1   gender                      1000 non-null   object 
 2   study_time_hours            1000 non-null   float64
 3   attendance_percent          1000 non-null   float64
 4   sleep_hours                 1000 non-null   float64
 5   parental_education          898 non-null    object 
 6   internet_access             1000 non-null   object 
 7   extracurricular_activities  1000 non-null   object 
 8   part_time_job               1000 non-null   object 
 9   previous_grade              1000 non-null   float64
 10  final_exam_score            1000 non-null   float64
 11  final_grade                 1000 non-null   object 
dtypes: float64(5), int64(1), object(6)
memory usage: 93.9+ KB


In [48]:
df.columns

Index(['student_id', 'gender', 'study_time_hours', 'attendance_percent',
       'sleep_hours', 'parental_education', 'internet_access',
       'extracurricular_activities', 'part_time_job', 'previous_grade',
       'final_exam_score', 'final_grade'],
      dtype='object')

In [49]:
df.isnull().sum()

,0
student_id,0
gender,0
study_time_hours,0
attendance_percent,0
sleep_hours,0
parental_education,102
internet_access,0
extracurricular_activities,0
part_time_job,0
previous_grade,0


In [50]:
df['parental_education'].value_counts(dropna=False)

,count
parental_education,
High School,356
Bachelors,308
Masters,184
NaN,102
PhD,50


In [51]:
df['parental_education'] = df['parental_education'].fillna('Unknown')

In [52]:
df['parental_education'].value_counts(dropna=False)

,count
parental_education,
High School,356
Bachelors,308
Masters,184
Unknown,102
PhD,50


In [53]:
df.drop('student_id',axis=1,inplace=True)

In [54]:
df.head()

,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,Male,2.6,77.5,8.0,Unknown,Yes,Yes,No,85.1,83.8,B
4,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D


In [65]:
target = 'final_exam_score'
y = df[target]
X = df.drop(columns=target, axis=1)

# now seperate train, test, val 60, 20, 20 percent

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

# ============================================================
# 1. One-hot encode categorical features
# ============================================================
categorical_cols = X_train.select_dtypes(include='object').columns

X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_val = pd.get_dummies(X_val, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Align columns - crucial for consistent feature sets after one-hot encoding
# especially if some categories are not present in all splits
train_cols = X_train.columns
X_val = X_val.reindex(columns=train_cols, fill_value=0)
X_test = X_test.reindex(columns=train_cols, fill_value=0)


# ============================================================
# 2. FIT SCALER ON TRAIN ONLY (prevents leakage)
# ============================================================

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
X_val_s = scaler.transform(X_val)

# ============================================================
# 3. TRAIN WITH EARLY STOPPING
# ============================================================
# Note: GBM's validation_fraction creates its own internal split from X_train.
# X_val above is for your external evaluation only, not used by early stopping.

from sklearn.ensemble import GradientBoostingRegressor # Changed to Regressor


# fix overfitting if occured

model = GradientBoostingRegressor(
    n_estimators=1000,
    learning_rate=0.05,       # Slower learning rate
    max_depth=3,              # Shallower trees (crucial for GBM)
    min_samples_leaf=5,       # Forces trees to generalize at the leaves
    validation_fraction=0.1,  # Give early stopping a bigger chunk of X_train to watch
    n_iter_no_change=20,      # Wait a bit longer before stopping
    tol=0.001,
    random_state=42
)

# model = GradientBoostingRegressor( # Changed to Regressor
#     n_estimators=1000,
#     learning_rate=0.1,
#     max_depth=5,
#     validation_fraction=0.1,
#     n_iter_no_change=10,
#     tol=0.001,
#     random_state=42
# )

model.fit(X_train_s, y_train)


train_score = model.score(X_train_s, y_train)
val_score = model.score(X_val_s, y_val)
test_score = model.score(X_test_s, y_test)

print(f"Stopped at {model.n_estimators_} iterations (of 1000 max)")
print(f"Train: {train_score:.3f}, Val: {val_score:.3f}, Test: {test_score:.3f}")
print(f"Train-Val gap: {train_score - val_score:.3f}")
print("Healthy" if train_score - val_score < 0.05 else "Overfitting detected")


Train: 600, Val: 200, Test: 200
Stopped at 250 iterations (of 1000 max)
Train: 0.958, Val: 0.898, Test: 0.904
Train-Val gap: 0.061
Overfitting detected


PyTorch Training Loop with Early Stopping

Explicit training loop with monitoring, early stopping, and best-model checkpointing

In [68]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class EarlyStopping:

  def __init__(self, patience=10, min_delta=0.001):
    self.patience = patience
    self.min_delta = min_delta
    self.counter = 0
    self.best_loss = float('inf')
    self.best_weights = None

  def check(self, val_loss, model):
    # The variable val_score is not defined here. It should be val_loss.
    if val_loss < self.best_loss - self.min_delta:
      self.best_loss = val_loss

      self.best_weights = {
          k : v.clone() for k, v in model.state_dict().items()
      }

      self.counter = 0
      return False # continue training
    self.counter += 1
    return self.counter >= self.patience # True = stop

  def restore(self, model):
    if self.best_weights:
      model.load_state_dict(self.best_weights)

# ============================================================
# TRAINING FUNCTION WITH FULL MONITORING
# ============================================================

def train_model(model, train_loader, val_loader, epochs=1000, lr = 0.001):
  # For a regression problem (final_exam_score is continuous),
  # CrossEntropyLoss is incorrect. MSELoss should be used.
  criterion = nn.MSELoss()
  # Adam with lr=0.001 - good default
  optimizer = torch.optim.Adam(model.parameters(), lr=lr)
  stopper = EarlyStopping(patience=10)
  history = {'train_loss':[], 'val_loss':[]}

  for epoch in range(epochs):
    model.train()
    total_train = 0
    for X_batch, y_batch in train_loader:
      optimizer.zero_grad() #clear old gradients
      outputs = model(X_batch) # forward passs
      # y_batch might need to be reshaped for MSELoss if outputs is 2D and y_batch is 1D
      loss = criterion(outputs.squeeze(), y_batch.float()) # compute loss
      loss.backward() #backward pass gradient
      optimizer.step() # update weights
      total_train += loss.item()
    avg_train = total_train / len(train_loader)

    model.eval()
    total_val = 0
    with torch.no_grad():
      for X_batch, y_batch in val_loader:
        outputs = model(X_batch)
        # y_batch might need to be reshaped for MSELoss if outputs is 2D and y_batch is 1D
        total_val += criterion(outputs.squeeze(), y_batch.float()).item()
    # len(total_val) is incorrect, should be len(val_loader)
    avg_val = total_val / len(val_loader)

    history['train_loss'].append(avg_train)
    history['val_loss'].append(avg_val)
    print(f"Epoch {epoch+1}: train={avg_train:.4f}, val={avg_val:.4f}")

    # Early stopping check
    if stopper.check(avg_val, model):
      print(f"Early stopping at epoch {epoch+1}!")
      # Typo: resotre should be restore
      stopper.restore(model) # roll back to best weights
      break
  return history

# ============================================================
# 2. CONVERT SKLEARN DATA TO PYTORCH TENSORS
# ============================================================
# Assuming X_train_s, X_val_s are already scaled from your previous code
# Assuming y_train, y_val are Pandas Series

# Convert to PyTorch float32 tensors
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32)

X_val_t = torch.tensor(X_val_s, dtype=torch.float32)
y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

# Wrap in DataLoaders (batch size of 32 is a good default)
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=32, shuffle=False)

# ============================================================
# 3. BUILD THE NETWORK AND TRAIN
# ============================================================
class StudentGradeNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1) # Outputting a single number (the exam score)
        )

    def forward(self, x):
        return self.net(x)

# Automatically set input size based on your one-hot encoded columns
input_features = X_train_t.shape[1]
model = StudentGradeNN(input_features)

print("Starting Neural Network Training...")
history = train_model(model, train_loader, val_loader, epochs=300, lr=0.01)

Starting Neural Network Training...
Epoch 1: train=6843.7164, val=6311.5684
Epoch 2: train=4631.4748, val=2269.9181
Epoch 3: train=764.4472, val=350.5196
Epoch 4: train=154.9796, val=103.2078
Epoch 5: train=67.4279, val=65.8410
Epoch 6: train=41.1718, val=47.7776
Epoch 7: train=30.5290, val=43.4224
Epoch 8: train=24.9914, val=38.9841
Epoch 9: train=21.7587, val=36.1246
Epoch 10: train=19.5106, val=33.4270
Epoch 11: train=17.5355, val=32.7850
Epoch 12: train=15.7672, val=29.7858
Epoch 13: train=14.5049, val=29.0101
Epoch 14: train=13.4484, val=26.9532
Epoch 15: train=12.5589, val=25.9358
Epoch 16: train=11.6359, val=25.5582
Epoch 17: train=11.3922, val=25.0102
Epoch 18: train=10.4577, val=23.8260
Epoch 19: train=10.0662, val=23.9256
Epoch 20: train=9.6544, val=21.9824
Epoch 21: train=9.1741, val=21.4035
Epoch 22: train=8.6757, val=21.0924
Epoch 23: train=8.5187, val=20.9900
Epoch 24: train=8.0098, val=20.3366
Epoch 25: train=7.8863, val=20.5410
Epoch 26: train=7.8076, val=20.3349
Epoch 

Learning Rate Finder

Automatically find the optimal learning rate using the LR range test

In [66]:
import torch
import torch.nn as nn
import numpy as np

# ============================================================
# LR RANGE TEST (Smith, 2015)
# Find optimal learning rate automatically
# ============================================================
def lr_range_test(model, train_loader, criterion,
                  lr_start=1e-7, lr_end=1.0, num_steps=100):
    """Train with exponentially increasing LR.
    Best LR = steepest downward slope (not the minimum)."""

    optimizer = torch.optim.SGD(model.parameters(), lr=lr_start)

    # Save initial weights to restore later
    init_state = {k: v.clone() for k, v in model.state_dict().items()}

    # Exponential schedule: lr_start -> lr_end over num_steps
    gamma = (lr_end / lr_start) ** (1 / num_steps)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(
        optimizer, gamma
    )

    lrs, losses = [], []
    best_loss = float('inf')

    for step, (X_batch, y_batch) in enumerate(train_loader):
        if step >= num_steps:
            break

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        scheduler.step()

        current_lr = optimizer.param_groups[0]['lr']
        current_loss = loss.item()

        # Stop if loss explodes (4x best = diverging)
        if current_loss > best_loss * 4:
            break
        best_loss = min(best_loss, current_loss)

        lrs.append(current_lr)
        losses.append(current_loss)

    # Restore original weights (test is non-destructive)
    model.load_state_dict(init_state)

    # Find steepest descent = best LR region
    smoothed = np.convolve(losses, np.ones(5)/5, mode='valid')
    gradients = np.gradient(smoothed)
    best_idx = np.argmin(gradients)
    suggested_lr = lrs[best_idx + 2]  # Offset for smoothing

    print(f"Suggested LR: {suggested_lr:.6f}")
    print(f"Tested range: {lrs[0]:.1e} to {lrs[-1]:.1e}")
    print(f"Tip: Use LR where loss drops fastest, not minimum")
    return lrs, losses, suggested_lr